In [14]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

data = pd.read_csv("data/FRED data.csv")

data["date"] = pd.to_datetime(data["date"])
data = data.sort_values("date")

In [13]:
#EDA
data.info()
data.describe()
data.isnull().sum()
data["recession"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 410 entries, 0 to 409
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   date                   410 non-null    datetime64[ns]
 1   unemployment           410 non-null    float64       
 2   cpi                    410 non-null    float64       
 3   industrial_production  410 non-null    float64       
 4   retail_sales           410 non-null    int64         
 5   consumer_sentiment     410 non-null    float64       
 6   recession              410 non-null    int64         
dtypes: datetime64[ns](1), float64(4), int64(2)
memory usage: 22.6 KB


recession
0    382
1     28
Name: count, dtype: int64

In [ ]:
#Feature Engineering
feature_cols = ["unemployment", "cpi", "industrial_production", "retail_sales", "consumer_sentiment"]

#Lag
for col in feature_cols:
    data[f"{col}_lag1"] = data[col].shift(1)
    data[f"{col}_lag3"] = data[col].shift(3)
    data[f"{col}_lag6"] = data[col].shift(6)

for col in feature_cols:
    data[f"{col}_change"] = data[col].pct_change()

data = data.dropna()

In [ ]:
#Training Models
X = data.drop(columns=["date", "recession"])
y = data["recession"]

split = int(len(data) * 0.8)
X_train = X.iloc[:split]
X_test = X.iloc[split:]
y_train = y.iloc[:split]
y_test = y.iloc[split:]

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)

lr_pred = lr.predict(X_test)
rf_pred = rf.predict(X_test)
gb_pred = gb.predict(X_test)

print("Logistic Regression")
print(classification_report(y_test, lr_pred))

print("Random Forest")
print(classification_report(y_test, rf_pred))

print("Gradient Boosting")
print(classification_report(y_test, gb_pred))

In [ ]:
#Evaluation
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, zero_division=0))
    print("Recall:", recall_score(y_test, y_pred, zero_division=0))
    print("F1 Score:", f1_score(y_test, y_pred, zero_division=0))
    print("ROC-AUC:", roc_auc_score(y_test, y_prob))

evaluate_model("Logistic Regression", lr, X_test, y_test)
evaluate_model("Random Forest", rf, X_test, y_test)
evaluate_model("Gradient Boosting", gb, X_test, y_test)

def show_confusion(name, model):
    y_pred = model.predict(X_test)
    print(f"\n{name}")
    print(confusion_matrix(y_test, y_pred))

show_confusion("Logistic Regression", lr)
show_confusion("Random Forest", rf)
show_confusion("Gradient Boosting", gb)

# Use best model (example: Gradient Boosting)
probs = gb.predict_proba(X_test)[:, 1]

plt.figure(figsize=(10,5))
plt.plot(data["date"].iloc[-len(probs):], probs, label="Predicted Probability")
plt.title("Recession Probability Over Time")
plt.xlabel("Date")
plt.ylabel("Probability")
plt.legend()
plt.show()